# Section 3 — Evolution across Gender

We explore how names are distributed between male and female,
and how this has changed over time.

In [ ]:
import altair as alt
import pandas as pd

alt.data_transformers.enable('json')

names = pd.read_csv("Names hints/dpt2020.csv", sep=";")
names = names[names.preusuel != '_PRENOMS_RARES']
names = names[names.dpt != 'XX']
names['annais'] = pd.to_numeric(names['annais'], errors='coerce')
names = names.dropna(subset=['annais'])
names['annais'] = names['annais'].astype(int)

MALE   = '#4C78A8'
FEMALE = '#F28E2B'

names.head()

## 3.a — Demographic butterfly chart

Each name gets a mirrored area chart: female counts grow to the right,
male counts to the left, with year on the vertical axis.
We show the names that are most evenly distributed between genders — the most sex-neutral names.

In [ ]:
# Find the most sex-neutral names (highest share of the minority gender)
totals = (names.groupby(['preusuel', 'sexe'])['nombre']
          .sum().unstack(fill_value=0))
totals.columns = ['male', 'female']
totals['total']   = totals['male'] + totals['female']
totals['balance'] = totals[['male', 'female']].min(axis=1) / totals['total']

neutral = totals[totals['total'] > 5_000].nlargest(6, 'balance').index.tolist()
print("Most sex-neutral names:", neutral)

by_yg = (names[names.preusuel.isin(neutral)]
         .groupby(['annais', 'preusuel', 'sexe'])['nombre'].sum().reset_index())
by_yg['gender']        = by_yg['sexe'].map({1: 'Male', 2: 'Female'})
by_yg['count_signed']  = by_yg.apply(
    lambda r: -r['nombre'] if r['sexe'] == 1 else r['nombre'], axis=1)

# orient='horizontal' → x is the value axis, y is the time axis
#   area silhouettes grow left (male) and right (female) from x=0
# scale domain [2020, 1900] reverses the year axis so recent years sit at the top
# stack=None draws each gender independently from x=0 (no stacking)
alt.Chart(by_yg).mark_area(opacity=0.7, interpolate='monotone', orient='horizontal').encode(
    y=alt.Y('annais:Q',
            scale=alt.Scale(domain=[2020, 1900]),
            axis=alt.Axis(format='d'),
            title='Year'),
    x=alt.X('count_signed:Q', stack=None, title='← Male | Female →'),
    color=alt.Color('gender:N',
                    scale=alt.Scale(domain=['Male', 'Female'], range=[MALE, FEMALE])),
    tooltip=['preusuel:N', 'annais:Q', 'gender:N',
             alt.Tooltip('nombre:Q', format=',')],
).facet(
    facet=alt.Facet('preusuel:N', title='Name'), columns=3
).resolve_scale(x='independent')

## 3.b — Diverging bar chart

Female totals are shown as positive bars (above the axis) and male totals as negative bars (below).
We use a symmetric log scale so that less common names remain visible alongside very popular ones.
Sex-neutral names produce roughly equal bars on both sides.

In [ ]:
N = 25
both = (names.groupby(['preusuel', 'sexe'])['nombre']
        .sum().unstack(fill_value=0))
both.columns = ['male', 'female']
both['total'] = both['male'] + both['female']
top = both.nlargest(N, 'total').reset_index()

male_df   = top[['preusuel', 'male']].rename(columns={'male': 'count'}).copy()
male_df['gender']       = 'Male'
male_df['count_signed'] = -male_df['count']

female_df = top[['preusuel', 'female']].rename(columns={'female': 'count'}).copy()
female_df['gender']       = 'Female'
female_df['count_signed'] = female_df['count']

diverging = pd.concat([male_df, female_df])

bars = alt.Chart(diverging).mark_bar().encode(
    x=alt.X('preusuel:N',
            sort=alt.EncodingSortField(field='count', op='max', order='descending'),
            axis=alt.Axis(labelAngle=-45), title='Name'),
    y=alt.Y('count_signed:Q',
            scale=alt.Scale(type='symlog', constant=1_000),
            title='← Male | Female →',
            axis=alt.Axis(format=',.0f')),
    color=alt.Color('gender:N',
                    scale=alt.Scale(domain=['Male', 'Female'], range=[MALE, FEMALE])),
    tooltip=['preusuel:N', 'gender:N', alt.Tooltip('count:Q', format=',')],
)

zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='black').encode(y='y:Q')

(bars + zero).properties(width=750, height=400)

## 3.c — Grouped alternating bar chart

The top 10 male names are paired with the top 10 female names by rank.
Both bars for the same rank appear side-by-side.
Drag the slider to see how both the names and their counts change over the years.

In [ ]:
N = 10
top_m = (names[names.sexe == 1].groupby('preusuel')['nombre']
         .sum().nlargest(N).reset_index())
top_m['rank']   = range(1, N + 1)
top_m['gender'] = 'Male'

top_f = (names[names.sexe == 2].groupby('preusuel')['nombre']
         .sum().nlargest(N).reset_index())
top_f['rank']   = range(1, N + 1)
top_f['gender'] = 'Female'

all_top  = pd.concat([top_m, top_f])
all_names = all_top['preusuel'].tolist()

annual = (names[names.preusuel.isin(all_names)]
          .groupby(['annais', 'preusuel', 'sexe'])['nombre'].sum().reset_index())
annual = annual.merge(all_top[['preusuel', 'rank', 'gender']], on='preusuel')

year_param = alt.param(
    name='year',
    bind=alt.binding_range(min=1900, max=2020, step=1, name='Year: '),
    value=2000,
)

bars = alt.Chart(annual).mark_bar().encode(
    x=alt.X('nombre:Q', title='Count'),
    y=alt.Y('rank:O', title='Rank'),
    yOffset=alt.YOffset('gender:N'),
    color=alt.Color('gender:N',
                    scale=alt.Scale(domain=['Male', 'Female'], range=[MALE, FEMALE])),
    tooltip=['preusuel:N', 'gender:N', 'rank:O',
             alt.Tooltip('nombre:Q', format=',')],
)

labels = alt.Chart(annual).mark_text(align='left', dx=4, fontSize=10).encode(
    x=alt.X('nombre:Q'),
    y=alt.Y('rank:O'),
    yOffset=alt.YOffset('gender:N'),
    text='preusuel:N',
    color=alt.value('black'),
)

(bars + labels).transform_filter(
    alt.datum.annais == year_param
).add_params(year_param).properties(width=500, height=400)